# Phase 3 — Model Training, Evaluation, and Selection

- **3A.** Inspect Phase 2 artifacts
- **3B.** Build persistence baselines
- **3C.** Train Ridge Regression
- **3D.** Train tree-based models
- **3E.** Select the best model
- **3F.** Focused validation error analysis
- **3G.** Final test evaluation
- **3H.** Save artifacts

## **3A.** Inspect and validate Phase 2 artifacts

Phase 2 produced the model-ready training, validation, and testing datasets.

Before training any model, this notebook verifies that:

- all required artifact files exist
- feature metadata can be loaded
- train, validation, and test schemas match
- the configured feature columns exist
- the target column exists
- identifiers and targets are not included as model inputs
- feature and target values are complete
- reference timestamps remain chronological
- forecast horizons cover 1 through 72
- no duplicate reference-time and horizon keys exist

No model is trained in this subphase.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().parent

TRAINING_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "training"
)

FEATURE_COLUMNS_PATH = (
    TRAINING_DATA_DIR
    / "feature_columns.json"
)

PHASE_2_REPORT_PATH = (
    TRAINING_DATA_DIR
    / "phase_2_validation_report.json"
)

TRAIN_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "train_dataset.parquet"
)

VALIDATION_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "validation_dataset.parquet"
)

TEST_DATASET_PATH = (
    TRAINING_DATA_DIR
    / "test_dataset.parquet"
)


print("Project root:", PROJECT_ROOT)
print("Training data directory:", TRAINING_DATA_DIR)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Training data directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training


In [2]:
required_artifact_paths = {
    "feature metadata": FEATURE_COLUMNS_PATH,
    "Phase 2 report": PHASE_2_REPORT_PATH,
    "training dataset": TRAIN_DATASET_PATH,
    "validation dataset": VALIDATION_DATASET_PATH,
    "testing dataset": TEST_DATASET_PATH,
}

artifact_status = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "path": str(artifact_path),
            "exists": artifact_path.exists(),
        }
        for artifact_name, artifact_path
        in required_artifact_paths.items()
    ]
)

display(artifact_status)


missing_artifacts = [
    artifact_name
    for artifact_name, artifact_path
    in required_artifact_paths.items()
    if not artifact_path.exists()
]

assert not missing_artifacts, (
    "Missing required Phase 2 artifacts: "
    f"{missing_artifacts}"
)

print("All required Phase 2 artifacts exist.")

,artifact,path,exists
0,feature metadata,/home/riyan/Riyan/projects/pearls-aqi-predicto...,True
1,Phase 2 report,/home/riyan/Riyan/projects/pearls-aqi-predicto...,True
2,training dataset,/home/riyan/Riyan/projects/pearls-aqi-predicto...,True
3,validation dataset,/home/riyan/Riyan/projects/pearls-aqi-predicto...,True
4,testing dataset,/home/riyan/Riyan/projects/pearls-aqi-predicto...,True


All required Phase 2 artifacts exist.


In [3]:
with open(
    FEATURE_COLUMNS_PATH,
    "r",
    encoding="utf-8",
) as file:
    feature_metadata = json.load(file)

with open(
    PHASE_2_REPORT_PATH,
    "r",
    encoding="utf-8",
) as file:
    phase_2_report = json.load(file)


MODEL_FEATURE_COLUMNS = feature_metadata[
    "feature_columns"
]

TARGET_COLUMN = feature_metadata[
    "target_column"
]

IDENTIFIER_COLUMNS = feature_metadata[
    "identifier_columns"
]


print("Number of model features:", len(MODEL_FEATURE_COLUMNS))
print("Target column:", TARGET_COLUMN)
print("Identifier columns:", IDENTIFIER_COLUMNS)
print(
    "Forecast horizon range:",
    feature_metadata["forecast_horizon_min"],
    "to",
    feature_metadata["forecast_horizon_max"],
)

Number of model features: 56
Target column: target_pm25_ug_m3
Identifier columns: ['reference_time', 'target_time']
Forecast horizon range: 1 to 72


In [4]:
print("Feature metadata:")
display(pd.Series(feature_metadata, dtype="object").to_frame("value"))

print("Phase 2 split summary:")
display(
    pd.DataFrame(
        phase_2_report["splits"]
    ).T
)

Feature metadata:


,value
feature_columns,"[pm25_current, pm25_lag_1h, pm25_lag_3h, pm25_..."
target_column,target_pm25_ug_m3
identifier_columns,"[reference_time, target_time]"
number_of_features,56
forecast_horizon_min,1
forecast_horizon_max,72


Phase 2 split summary:


,rows,unique_reference_times,reference_start,reference_end,target_start,target_end
train,364798,5153,2025-07-09T00:00:00+00:00,2026-03-18T11:00:00+00:00,2025-07-09T01:00:00+00:00,2026-03-21T11:00:00+00:00
validation,71256,1047,2026-03-21T12:00:00+00:00,2026-05-27T21:00:00+00:00,2026-03-21T13:00:00+00:00,2026-05-30T21:00:00+00:00
test,76813,1121,2026-05-30T22:00:00+00:00,2026-07-23T22:00:00+00:00,2026-05-30T23:00:00+00:00,2026-07-23T23:00:00+00:00


In [5]:
train_df = pd.read_parquet(
    TRAIN_DATASET_PATH
)

validation_df = pd.read_parquet(
    VALIDATION_DATASET_PATH
)

test_df = pd.read_parquet(
    TEST_DATASET_PATH
)


print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (364798, 59)
Validation shape: (71256, 59)
Testing shape: (76813, 59)


In [6]:
train_columns = train_df.columns.tolist()
validation_columns = validation_df.columns.tolist()
test_columns = test_df.columns.tolist()


assert train_columns == validation_columns
assert train_columns == test_columns


assert train_df.dtypes.equals(
    validation_df.dtypes
)

assert train_df.dtypes.equals(
    test_df.dtypes
)


print("Train, validation, and test schemas match.")

Train, validation, and test schemas match.


In [7]:
schema_df = pd.DataFrame(
    {
        "column": train_df.columns,
        "dtype": train_df.dtypes.astype(str).values,
        "is_model_feature": [
            column in MODEL_FEATURE_COLUMNS
            for column in train_df.columns
        ],
        "is_target": [
            column == TARGET_COLUMN
            for column in train_df.columns
        ],
        "is_identifier": [
            column in IDENTIFIER_COLUMNS
            for column in train_df.columns
        ],
    }
)

display(schema_df)

,column,dtype,is_model_feature,is_target,is_identifier
0,reference_time,"datetime64[us, UTC]",False,False,True
1,target_time,"datetime64[us, UTC]",False,False,True
2,pm25_current,float64,True,False,False
3,pm25_lag_1h,float64,True,False,False
4,pm25_lag_3h,float64,True,False,False
5,pm25_lag_6h,float64,True,False,False
6,pm25_lag_12h,float64,True,False,False
7,pm25_lag_24h,float64,True,False,False
8,pm25_mean_3h,float64,True,False,False
9,pm25_mean_6h,float64,True,False,False


In [8]:
missing_model_features = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column not in train_df.columns
]

assert not missing_model_features, (
    "Missing model feature columns: "
    f"{missing_model_features}"
)

assert TARGET_COLUMN in train_df.columns

for identifier_column in IDENTIFIER_COLUMNS:
    assert identifier_column in train_df.columns


assert TARGET_COLUMN not in MODEL_FEATURE_COLUMNS

for identifier_column in IDENTIFIER_COLUMNS:
    assert identifier_column not in MODEL_FEATURE_COLUMNS

assert "forecast_horizon_hours" in MODEL_FEATURE_COLUMNS


print("Feature contract validation passed.")

Feature contract validation passed.


In [9]:
future_pm25_feature_columns = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column.startswith("target_pm25")
]

assert not future_pm25_feature_columns, (
    "Future PM2.5-derived features found: "
    f"{future_pm25_feature_columns}"
)

print("No future PM2.5-derived feature is included.")

No future PM2.5-derived feature is included.


## **3B.** Persistence baselines

Before training machine-learning models, simple persistence methods are
evaluated.

A trained model is only useful when it performs better than straightforward
rules based on recent PM2.5 observations.

Two baselines are used:

### Current-value persistence

Predict that future PM2.5 will remain equal to the PM2.5 concentration observed
at the reference timestamp.

`prediction = pm25_current`

This baseline is often strong for short forecast horizons because pollution
usually changes gradually.

### Previous-day persistence

Predict future PM2.5 using the PM2.5 value observed 24 hours before the
reference timestamp.

`prediction = pm25_lag_24h`

This baseline provides a simple daily-pattern comparison.

The validation dataset is used for baseline comparison. The test dataset
remains untouched until the best machine-learning model has been selected.

In [11]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

### Evaluation metrics

The baselines are evaluated using:

- **MAE** — average absolute prediction error
- **RMSE** — penalizes larger errors more strongly
- **R²** — measures how much variation in PM2.5 is explained

Lower MAE and RMSE are better.

Higher R² is better.

In [12]:
def calculate_regression_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
) -> dict[str, float]:
    """Calculate standard regression metrics."""

    return {
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred,
                )
            )
        ),
        "r2": float(
            r2_score(
                y_true,
                y_pred,
            )
        ),
    }

In [13]:
baseline_validation_df = validation_df[
    [
        "reference_time",
        "target_time",
        "forecast_horizon_hours",
        TARGET_COLUMN,
        "pm25_current",
        "pm25_lag_24h",
    ]
].copy()


baseline_validation_df[
    "prediction_current_persistence"
] = baseline_validation_df["pm25_current"]

baseline_validation_df[
    "prediction_previous_day"
] = baseline_validation_df["pm25_lag_24h"]


print("Validation baseline rows:", len(baseline_validation_df))

Validation baseline rows: 71256


In [14]:
baseline_missing_summary = (
    baseline_validation_df[
        [
            TARGET_COLUMN,
            "prediction_current_persistence",
            "prediction_previous_day",
        ]
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(baseline_missing_summary)


assert (
    baseline_missing_summary[
        "missing_count"
    ].sum()
    == 0
)

print("Baseline prediction inputs are complete.")

,missing_count
target_pm25_ug_m3,0
prediction_current_persistence,0
prediction_previous_day,0


Baseline prediction inputs are complete.


In [15]:
y_validation = baseline_validation_df[
    TARGET_COLUMN
]

current_persistence_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=baseline_validation_df[
            "prediction_current_persistence"
        ],
    )
)

previous_day_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=baseline_validation_df[
            "prediction_previous_day"
        ],
    )
)


baseline_overall_metrics = pd.DataFrame(
    [
        {
            "model": "current_persistence",
            **current_persistence_metrics,
        },
        {
            "model": "previous_day_persistence",
            **previous_day_metrics,
        },
    ]
).sort_values("rmse").reset_index(drop=True)

display(baseline_overall_metrics)

,model,mae,rmse,r2
0,current_persistence,6.621121,10.506971,-0.161334
1,previous_day_persistence,7.942474,11.887120,-0.486467


### Baseline performance by forecast distance

Prediction difficulty usually increases as the forecast horizon becomes
longer.

Metrics are therefore calculated for five horizon groups:

- 1–6 hours
- 7–12 hours
- 13–24 hours
- 25–48 hours
- 49–72 hours

This shows whether a baseline is strong only for immediate forecasts or remains
competitive across the full 72-hour period.

In [16]:
HORIZON_GROUPS = {
    "1-6h": (1, 6),
    "7-12h": (7, 12),
    "13-24h": (13, 24),
    "25-48h": (25, 48),
    "49-72h": (49, 72),
}

In [17]:
def assign_horizon_group(
    horizon: int,
) -> str:
    """Map a forecast horizon to its evaluation group."""

    for group_name, (
        minimum_horizon,
        maximum_horizon,
    ) in HORIZON_GROUPS.items():
        if (
            minimum_horizon
            <= horizon
            <= maximum_horizon
        ):
            return group_name

    raise ValueError(
        f"Unsupported forecast horizon: {horizon}"
    )


baseline_validation_df[
    "horizon_group"
] = baseline_validation_df[
    "forecast_horizon_hours"
].map(assign_horizon_group)


print(
    baseline_validation_df[
        "horizon_group"
    ].value_counts()
)

horizon_group
25-48h    23709
49-72h    23065
13-24h    12163
1-6h       6190
7-12h      6129
Name: count, dtype: int64


In [18]:
baseline_group_metrics_records = []

baseline_prediction_columns = {
    "current_persistence": (
        "prediction_current_persistence"
    ),
    "previous_day_persistence": (
        "prediction_previous_day"
    ),
}


for model_name, prediction_column in (
    baseline_prediction_columns.items()
):
    for group_name in HORIZON_GROUPS:
        group_df = baseline_validation_df.loc[
            baseline_validation_df[
                "horizon_group"
            ].eq(group_name)
        ]

        group_metrics = calculate_regression_metrics(
            y_true=group_df[TARGET_COLUMN],
            y_pred=group_df[prediction_column],
        )

        baseline_group_metrics_records.append(
            {
                "model": model_name,
                "horizon_group": group_name,
                "rows": len(group_df),
                **group_metrics,
            }
        )


baseline_group_metrics_df = pd.DataFrame(
    baseline_group_metrics_records
)

display(baseline_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,current_persistence,1-6h,6190,3.648142,6.990142,0.516136
1,current_persistence,7-12h,6129,5.131832,8.653139,0.268671
2,current_persistence,13-24h,12163,6.353498,10.294232,-0.024546
3,current_persistence,25-48h,23709,7.049690,10.982410,-0.354651
4,current_persistence,49-72h,23065,7.515322,11.321681,-0.384726
5,previous_day_persistence,1-6h,6190,7.476559,11.597199,-0.331856
6,previous_day_persistence,7-12h,6129,7.659896,11.973005,-0.400140
7,previous_day_persistence,13-24h,12163,7.970936,12.161715,-0.429991
8,previous_day_persistence,25-48h,23709,8.023101,11.907233,-0.592407
9,previous_day_persistence,49-72h,23065,8.044713,11.773361,-0.497417


In [19]:
assert len(baseline_overall_metrics) == 2

assert (
    baseline_overall_metrics[
        ["mae", "rmse", "r2"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert len(baseline_group_metrics_df) == 10

assert set(
    baseline_group_metrics_df[
        "horizon_group"
    ].unique()
) == set(HORIZON_GROUPS.keys())

assert (
    baseline_group_metrics_df[
        ["mae", "rmse", "r2"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

print("Validation persistence baselines evaluated successfully.")

Validation persistence baselines evaluated successfully.


## **3C.** Ridge Regression

Ridge Regression is the first trained machine-learning baseline.

It models the target as a linear combination of the 56 input features while
applying L2 regularization to reduce excessively large coefficients.

This model helps answer an important question:

> Can the engineered PM2.5, weather, time, and horizon features explain future
> PM2.5 using a relatively simple linear relationship?

The model is placed inside a scikit-learn pipeline:

1. `StandardScaler` learns feature means and standard deviations from the
   training set.
2. `Ridge` learns the regression coefficients from the scaled training data.

The validation set is only transformed and evaluated. It is not used to fit
the scaler or model.

In [20]:
from time import perf_counter

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Prepare model inputs

The ordered feature list from `feature_columns.json` remains the source of
truth.

Using the same ordered list for training and validation prevents accidental
column reordering.

The test dataset remains untouched.

In [21]:
X_train = train_df[MODEL_FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_validation = validation_df[MODEL_FEATURE_COLUMNS]
y_validation = validation_df[TARGET_COLUMN]


print("Training feature shape:", X_train.shape)
print("Training target shape:", y_train.shape)

print("Validation feature shape:", X_validation.shape)
print("Validation target shape:", y_validation.shape)

Training feature shape: (364798, 56)
Training target shape: (364798,)
Validation feature shape: (71256, 56)
Validation target shape: (71256,)


In [22]:
assert X_train.columns.tolist() == MODEL_FEATURE_COLUMNS
assert X_validation.columns.tolist() == MODEL_FEATURE_COLUMNS

assert X_train.isna().sum().sum() == 0
assert X_validation.isna().sum().sum() == 0

assert y_train.isna().sum() == 0
assert y_validation.isna().sum() == 0

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)

print("Ridge model inputs validated.")

Ridge model inputs validated.


### Build the Ridge pipeline

The initial Ridge model uses `alpha=1.0`.

`alpha` controls regularization strength:

- smaller values behave more like ordinary linear regression
- larger values shrink coefficients more strongly

We begin with one standard configuration rather than performing a large
hyperparameter search.

In [23]:
RIDGE_ALPHA = 1.0

ridge_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            Ridge(
                alpha=RIDGE_ALPHA,
                solver="auto",
            ),
        ),
    ]
)

ridge_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [24]:
ridge_training_start = perf_counter()

ridge_pipeline.fit(
    X_train,
    y_train,
)

ridge_training_seconds = (
    perf_counter()
    - ridge_training_start
)


print(
    f"Ridge training completed in "
    f"{ridge_training_seconds:.2f} seconds."
)

Ridge training completed in 1.71 seconds.


### Generate validation predictions

The fitted pipeline applies the training-set scaler to validation features and
then generates one PM2.5 prediction for every validation row.

Predictions are evaluated only against the validation target.

In [25]:
ridge_prediction_start = perf_counter()

ridge_validation_predictions = (
    ridge_pipeline.predict(
        X_validation
    )
)

ridge_prediction_seconds = (
    perf_counter()
    - ridge_prediction_start
)


print(
    "Number of validation predictions:",
    len(ridge_validation_predictions),
)

print(
    f"Prediction completed in "
    f"{ridge_prediction_seconds:.2f} seconds."
)

Number of validation predictions: 71256
Prediction completed in 0.07 seconds.


In [26]:
ridge_prediction_summary = pd.Series(
    ridge_validation_predictions,
    name="ridge_prediction",
).describe()

display(
    ridge_prediction_summary.to_frame()
)

print(
    "Negative predictions:",
    int(
        (
            ridge_validation_predictions < 0
        ).sum()
    ),
)

,ridge_prediction
count,71256.000000
mean,13.132456
std,14.361287
min,-57.801576
25%,2.744857
50%,11.159410
75%,22.255403
max,77.497770


Negative predictions: 11754


### Evaluate overall Ridge performance

The same MAE, RMSE, and R² metrics used for persistence baselines are applied
to Ridge Regression.

The main comparison is whether Ridge reduces validation RMSE below current
persistence.

In [27]:
ridge_validation_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=ridge_validation_predictions,
    )
)

ridge_overall_metrics_df = pd.DataFrame(
    [
        {
            "model": "ridge",
            "alpha": RIDGE_ALPHA,
            "training_seconds": (
                ridge_training_seconds
            ),
            "prediction_seconds": (
                ridge_prediction_seconds
            ),
            **ridge_validation_metrics,
        }
    ]
)

display(ridge_overall_metrics_df)

,model,alpha,training_seconds,prediction_seconds,mae,rmse,r2
0,ridge,1.0,1.709114,0.065426,9.779916,12.994571,-0.77634


In [28]:
ridge_baseline_comparison_df = pd.concat(
    [
        baseline_overall_metrics[
            [
                "model",
                "mae",
                "rmse",
                "r2",
            ]
        ],
        ridge_overall_metrics_df[
            [
                "model",
                "mae",
                "rmse",
                "r2",
            ]
        ],
    ],
    ignore_index=True,
).sort_values(
    "rmse"
).reset_index(drop=True)

display(ridge_baseline_comparison_df)

,model,mae,rmse,r2
0,current_persistence,6.621121,10.506971,-0.161334
1,previous_day_persistence,7.942474,11.887120,-0.486467
2,ridge,9.779916,12.994571,-0.776340


In [29]:
ridge_validation_results_df = (
    validation_df[
        [
            "reference_time",
            "target_time",
            "forecast_horizon_hours",
            TARGET_COLUMN,
        ]
    ]
    .copy()
)

ridge_validation_results_df[
    "prediction"
] = ridge_validation_predictions

ridge_validation_results_df[
    "horizon_group"
] = ridge_validation_results_df[
    "forecast_horizon_hours"
].map(assign_horizon_group)

In [30]:
ridge_group_metric_records = []

for group_name in HORIZON_GROUPS:
    group_df = (
        ridge_validation_results_df.loc[
            ridge_validation_results_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    group_metrics = calculate_regression_metrics(
        y_true=group_df[TARGET_COLUMN],
        y_pred=group_df["prediction"],
    )

    ridge_group_metric_records.append(
        {
            "model": "ridge",
            "horizon_group": group_name,
            "rows": len(group_df),
            **group_metrics,
        }
    )


ridge_group_metrics_df = pd.DataFrame(
    ridge_group_metric_records
)

display(ridge_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,ridge,1-6h,6190,9.016432,11.763965,-0.370435
1,ridge,7-12h,6129,9.832274,12.705341,-0.576659
2,ridge,13-24h,12163,9.461648,12.342200,-0.472750
3,ridge,25-48h,23709,10.011894,13.425849,-1.024490
4,ridge,49-72h,23065,9.900279,13.267045,-0.901474


In [31]:
ridge_baseline_group_comparison_df = (
    pd.concat(
        [
            baseline_group_metrics_df,
            ridge_group_metrics_df,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_group",
            "rmse",
        ]
    )
    .reset_index(drop=True)
)

display(ridge_baseline_group_comparison_df)

,model,horizon_group,rows,mae,rmse,r2
0,current_persistence,1-6h,6190,3.648142,6.990142,0.516136
1,previous_day_persistence,1-6h,6190,7.476559,11.597199,-0.331856
2,ridge,1-6h,6190,9.016432,11.763965,-0.370435
3,current_persistence,13-24h,12163,6.353498,10.294232,-0.024546
4,previous_day_persistence,13-24h,12163,7.970936,12.161715,-0.429991
5,ridge,13-24h,12163,9.461648,12.342200,-0.472750
6,current_persistence,25-48h,23709,7.049690,10.982410,-0.354651
7,previous_day_persistence,25-48h,23709,8.023101,11.907233,-0.592407
8,ridge,25-48h,23709,10.011894,13.425849,-1.024490
9,current_persistence,49-72h,23065,7.515322,11.321681,-0.384726


In [32]:
assert len(
    ridge_validation_predictions
) == len(validation_df)

assert np.isfinite(
    ridge_validation_predictions
).all()

assert len(ridge_group_metrics_df) == 5

assert (
    ridge_group_metrics_df[
        ["mae", "rmse", "r2"]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert set(
    ridge_group_metrics_df[
        "horizon_group"
    ]
) == set(HORIZON_GROUPS)

print(
    "Ridge Regression validation evaluation passed."
)

Ridge Regression validation evaluation passed.


## **3D.** Histogram Gradient Boosting

Ridge Regression could not capture the nonlinear relationships in the data and
performed worse than the persistence baselines.

Histogram Gradient Boosting is now evaluated because it can learn nonlinear
relationships and interactions between:

- recent PM2.5 history
- rolling pollution patterns
- weather conditions
- time features
- forecast horizon

Unlike Ridge Regression, feature scaling is not required for this tree-based
model.

The model is trained only on the chronological training split and evaluated on
the existing validation split.

In [33]:
from sklearn.ensemble import HistGradientBoostingRegressor

### Initial model configuration

The first configuration is intentionally practical rather than heavily tuned.

Key parameters:

- `learning_rate=0.05` uses gradual boosting updates
- `max_iter=300` allows up to 300 boosting rounds
- `max_leaf_nodes=31` limits tree complexity
- `min_samples_leaf=30` reduces overly specific leaves
- `l2_regularization=1.0` adds regularization
- `early_stopping=False` prevents an internal random validation split

Hyperparameter tuning will only be considered if the initial model shows
promising validation performance.

In [34]:
RANDOM_SEED = 42

hist_gradient_boosting_model = (
    HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=RANDOM_SEED,
    )
)

hist_gradient_boosting_model

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",300
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",30
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",False
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0


In [35]:
assert X_train.columns.tolist() == MODEL_FEATURE_COLUMNS
assert X_validation.columns.tolist() == MODEL_FEATURE_COLUMNS

assert X_train.isna().sum().sum() == 0
assert X_validation.isna().sum().sum() == 0

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_validation.to_numpy()).all()

print("Histogram Gradient Boosting inputs validated.")

Histogram Gradient Boosting inputs validated.


### Train the model

Training time is recorded because deployment suitability depends not only on
accuracy but also on computational cost.

The test dataset remains untouched.

In [36]:
hist_training_start = perf_counter()

hist_gradient_boosting_model.fit(
    X_train,
    y_train,
)

hist_training_seconds = (
    perf_counter()
    - hist_training_start
)

print(
    "Histogram Gradient Boosting training completed in "
    f"{hist_training_seconds:.2f} seconds."
)

print(
    "Boosting iterations completed:",
    hist_gradient_boosting_model.n_iter_,
)

Histogram Gradient Boosting training completed in 33.58 seconds.
Boosting iterations completed: 300


In [37]:
hist_prediction_start = perf_counter()

hist_validation_predictions = (
    hist_gradient_boosting_model.predict(
        X_validation
    )
)

hist_prediction_seconds = (
    perf_counter()
    - hist_prediction_start
)

print(
    "Validation predictions:",
    len(hist_validation_predictions),
)

print(
    "Prediction completed in "
    f"{hist_prediction_seconds:.2f} seconds."
)

Validation predictions: 71256
Prediction completed in 0.74 seconds.


### Inspect prediction behavior

PM2.5 cannot physically be negative.

The raw prediction distribution is inspected before applying any clipping so
that the model can be evaluated honestly in its original form.

In [38]:
hist_prediction_summary = pd.Series(
    hist_validation_predictions,
    name="hist_gradient_boosting_prediction",
).describe()

display(
    hist_prediction_summary.to_frame()
)

hist_negative_prediction_count = int(
    (
        hist_validation_predictions < 0
    ).sum()
)

print(
    "Negative predictions:",
    hist_negative_prediction_count,
)

,hist_gradient_boosting_prediction
count,71256.000000
mean,17.133745
std,7.583075
min,4.437495
25%,11.730243
50%,16.540002
75%,20.238701
max,101.819267


Negative predictions: 0


In [39]:
hist_validation_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=hist_validation_predictions,
    )
)

hist_overall_metrics_df = pd.DataFrame(
    [
        {
            "model": "hist_gradient_boosting",
            "training_seconds": (
                hist_training_seconds
            ),
            "prediction_seconds": (
                hist_prediction_seconds
            ),
            "iterations": (
                hist_gradient_boosting_model.n_iter_
            ),
            **hist_validation_metrics,
        }
    ]
)

display(hist_overall_metrics_df)

,model,training_seconds,prediction_seconds,iterations,mae,rmse,r2
0,hist_gradient_boosting,33.58392,0.741567,300,7.176154,9.827731,-0.016035


In [40]:
validation_model_comparison_df = pd.concat(
    [
        baseline_overall_metrics[
            [
                "model",
                "mae",
                "rmse",
                "r2",
            ]
        ],
        ridge_overall_metrics_df[
            [
                "model",
                "mae",
                "rmse",
                "r2",
            ]
        ],
        hist_overall_metrics_df[
            [
                "model",
                "mae",
                "rmse",
                "r2",
            ]
        ],
    ],
    ignore_index=True,
).sort_values(
    "rmse"
).reset_index(drop=True)

display(validation_model_comparison_df)

,model,mae,rmse,r2
0,hist_gradient_boosting,7.176154,9.827731,-0.016035
1,current_persistence,6.621121,10.506971,-0.161334
2,previous_day_persistence,7.942474,11.887120,-0.486467
3,ridge,9.779916,12.994571,-0.776340


### Evaluate performance across forecast horizons

Overall metrics can hide poor performance at specific forecast distances.

The model is therefore evaluated across the same five horizon groups used for
the persistence and Ridge baselines.

In [41]:
hist_validation_results_df = validation_df[
    [
        "reference_time",
        "target_time",
        "forecast_horizon_hours",
        TARGET_COLUMN,
    ]
].copy()

hist_validation_results_df[
    "prediction"
] = hist_validation_predictions

hist_validation_results_df[
    "horizon_group"
] = hist_validation_results_df[
    "forecast_horizon_hours"
].map(assign_horizon_group)

In [42]:
hist_group_metric_records = []

for group_name in HORIZON_GROUPS:
    group_df = hist_validation_results_df.loc[
        hist_validation_results_df[
            "horizon_group"
        ].eq(group_name)
    ]

    group_metrics = calculate_regression_metrics(
        y_true=group_df[TARGET_COLUMN],
        y_pred=group_df["prediction"],
    )

    hist_group_metric_records.append(
        {
            "model": "hist_gradient_boosting",
            "horizon_group": group_name,
            "rows": len(group_df),
            **group_metrics,
        }
    )

hist_group_metrics_df = pd.DataFrame(
    hist_group_metric_records
)

display(hist_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,hist_gradient_boosting,1-6h,6190,6.691069,9.356443,0.133093
1,hist_gradient_boosting,7-12h,6129,6.869678,9.579014,0.103796
2,hist_gradient_boosting,13-24h,12163,7.160805,9.813267,0.068954
3,hist_gradient_boosting,25-48h,23709,7.315578,9.934614,-0.108496
4,hist_gradient_boosting,49-72h,23065,7.252553,9.913220,-0.061626


In [43]:
all_group_metrics_df = (
    pd.concat(
        [
            baseline_group_metrics_df,
            ridge_group_metrics_df,
            hist_group_metrics_df,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_group",
            "rmse",
        ]
    )
    .reset_index(drop=True)
)

display(all_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,current_persistence,1-6h,6190,3.648142,6.990142,0.516136
1,hist_gradient_boosting,1-6h,6190,6.691069,9.356443,0.133093
2,previous_day_persistence,1-6h,6190,7.476559,11.597199,-0.331856
3,ridge,1-6h,6190,9.016432,11.763965,-0.370435
4,hist_gradient_boosting,13-24h,12163,7.160805,9.813267,0.068954
5,current_persistence,13-24h,12163,6.353498,10.294232,-0.024546
6,previous_day_persistence,13-24h,12163,7.970936,12.161715,-0.429991
7,ridge,13-24h,12163,9.461648,12.342200,-0.472750
8,hist_gradient_boosting,25-48h,23709,7.315578,9.934614,-0.108496
9,current_persistence,25-48h,23709,7.049690,10.982410,-0.354651


## **3D.2** XGBoost Regression

Histogram Gradient Boosting improved overall validation RMSE, particularly for
medium- and long-range forecasts, but current-value persistence remained
stronger for short horizons.

XGBoost is evaluated as the next nonlinear model because it can learn complex
interactions between:

- recent PM2.5 history
- rolling and change features
- current and target-hour weather
- cyclical time features
- forecast horizon

The model is trained only on the chronological training split.

The chronological validation split is used for early stopping so the model can
stop when additional boosting rounds no longer improve validation RMSE.

The test dataset remains untouched.

In [44]:
from xgboost import XGBRegressor

### Initial XGBoost configuration

The initial configuration is intentionally limited and practical.

Important settings:

- `objective="reg:squarederror"` for continuous PM2.5 prediction
- `tree_method="hist"` for efficient training on tabular data
- `learning_rate=0.05` for gradual boosting updates
- `max_depth=6` to capture nonlinear relationships without very deep trees
- `subsample=0.8` to reduce overfitting
- `colsample_bytree=0.8` to use a subset of features per tree
- L1 and L2 regularization for additional control
- early stopping based on chronological validation RMSE

A high maximum tree count is provided, but early stopping may finish training
much earlier.

In [45]:
XGBOOST_RANDOM_SEED = 42

xgboost_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="rmse",
    early_stopping_rounds=75,
    random_state=XGBOOST_RANDOM_SEED,
    n_jobs=-1,
)

xgboost_model

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",75
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [46]:
assert X_train.columns.tolist() == MODEL_FEATURE_COLUMNS
assert X_validation.columns.tolist() == MODEL_FEATURE_COLUMNS

assert X_train.isna().sum().sum() == 0
assert X_validation.isna().sum().sum() == 0

assert np.isfinite(X_train.to_numpy()).all()
assert np.isfinite(X_validation.to_numpy()).all()

assert y_train.isna().sum() == 0
assert y_validation.isna().sum() == 0

print("XGBoost model inputs validated.")

XGBoost model inputs validated.


### Train using chronological validation

The validation split is passed through `eval_set`.

Early stopping watches validation RMSE and stops when it does not improve for
75 consecutive boosting rounds.

This is limited model selection using validation data. The test set is not used
during training.

In [47]:
xgboost_training_start = perf_counter()

xgboost_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_validation, y_validation),
    ],
    verbose=False,
)

xgboost_training_seconds = (
    perf_counter()
    - xgboost_training_start
)

print(
    "XGBoost training completed in "
    f"{xgboost_training_seconds:.2f} seconds."
)

print(
    "Best boosting iteration:",
    xgboost_model.best_iteration,
)

print(
    "Best validation score:",
    xgboost_model.best_score,
)

XGBoost training completed in 40.02 seconds.
Best boosting iteration: 351
Best validation score: 9.63967297783432


In [48]:
xgboost_evaluation_results = (
    xgboost_model.evals_result()
)

validation_rmse_history = (
    xgboost_evaluation_results[
        "validation_0"
    ]["rmse"]
)

print(
    "Boosting rounds completed:",
    len(validation_rmse_history),
)

print(
    "Initial validation RMSE:",
    validation_rmse_history[0],
)

print(
    "Best validation RMSE:",
    min(validation_rmse_history),
)

print(
    "Final recorded validation RMSE:",
    validation_rmse_history[-1],
)

Boosting rounds completed: 427
Initial validation RMSE: 32.10008700202416
Best validation RMSE: 9.63967297783432
Final recorded validation RMSE: 9.723627657597193


### Generate XGBoost validation predictions

Predictions are generated using the best boosting iteration selected through
early stopping.

The raw predictions are inspected before applying any physical lower bound.

In [49]:
xgboost_prediction_start = perf_counter()

xgboost_validation_predictions = (
    xgboost_model.predict(
        X_validation
    )
)

xgboost_prediction_seconds = (
    perf_counter()
    - xgboost_prediction_start
)

print(
    "Validation predictions:",
    len(xgboost_validation_predictions),
)

print(
    "Prediction completed in "
    f"{xgboost_prediction_seconds:.2f} seconds."
)

Validation predictions: 71256
Prediction completed in 0.46 seconds.


In [50]:
xgboost_prediction_summary = pd.Series(
    xgboost_validation_predictions,
    name="xgboost_prediction",
).describe()

display(
    xgboost_prediction_summary.to_frame()
)

xgboost_negative_prediction_count = int(
    (
        xgboost_validation_predictions < 0
    ).sum()
)

print(
    "Negative predictions:",
    xgboost_negative_prediction_count,
)

,xgboost_prediction
count,71256.000000
mean,17.307823
std,7.133305
min,0.932509
25%,11.996804
50%,17.145962
75%,21.240597
max,87.793289


Negative predictions: 0


In [51]:
xgboost_validation_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=xgboost_validation_predictions,
    )
)

xgboost_overall_metrics_df = pd.DataFrame(
    [
        {
            "model": "xgboost",
            "training_seconds": (
                xgboost_training_seconds
            ),
            "prediction_seconds": (
                xgboost_prediction_seconds
            ),
            "best_iteration": int(
                xgboost_model.best_iteration
            ),
            "negative_predictions": (
                xgboost_negative_prediction_count
            ),
            **xgboost_validation_metrics,
        }
    ]
)

display(xgboost_overall_metrics_df)

,model,training_seconds,prediction_seconds,best_iteration,negative_predictions,mae,rmse,r2
0,xgboost,40.024134,0.461011,351,0,7.182968,9.639673,0.022478


In [52]:
validation_model_comparison_df = (
    pd.concat(
        [
            baseline_overall_metrics[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
            ridge_overall_metrics_df[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
            hist_overall_metrics_df[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
            xgboost_overall_metrics_df[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
        ],
        ignore_index=True,
    )
    .sort_values("rmse")
    .reset_index(drop=True)
)

display(validation_model_comparison_df)

,model,mae,rmse,r2
0,xgboost,7.182968,9.639673,0.022478
1,hist_gradient_boosting,7.176154,9.827731,-0.016035
2,current_persistence,6.621121,10.506971,-0.161334
3,previous_day_persistence,7.942474,11.887120,-0.486467
4,ridge,9.779916,12.994571,-0.776340


In [53]:
xgboost_validation_results_df = (
    validation_df[
        [
            "reference_time",
            "target_time",
            "forecast_horizon_hours",
            TARGET_COLUMN,
        ]
    ]
    .copy()
)

xgboost_validation_results_df[
    "prediction"
] = xgboost_validation_predictions

xgboost_validation_results_df[
    "horizon_group"
] = xgboost_validation_results_df[
    "forecast_horizon_hours"
].map(assign_horizon_group)

In [54]:
xgboost_group_metric_records = []

for group_name in HORIZON_GROUPS:
    group_df = (
        xgboost_validation_results_df.loc[
            xgboost_validation_results_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    group_metrics = calculate_regression_metrics(
        y_true=group_df[TARGET_COLUMN],
        y_pred=group_df["prediction"],
    )

    xgboost_group_metric_records.append(
        {
            "model": "xgboost",
            "horizon_group": group_name,
            "rows": len(group_df),
            **group_metrics,
        }
    )


xgboost_group_metrics_df = pd.DataFrame(
    xgboost_group_metric_records
)

display(xgboost_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,xgboost,1-6h,6190,6.580459,9.119446,0.176454
1,xgboost,7-12h,6129,6.828526,9.412145,0.134748
2,xgboost,13-24h,12163,7.150829,9.623150,0.104680
3,xgboost,25-48h,23709,7.279134,9.689118,-0.054389
4,xgboost,49-72h,23065,7.356946,9.791816,-0.035782


In [55]:
all_group_metrics_df = (
    pd.concat(
        [
            baseline_group_metrics_df,
            ridge_group_metrics_df,
            hist_group_metrics_df,
            xgboost_group_metrics_df,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_group",
            "rmse",
        ]
    )
    .reset_index(drop=True)
)

display(all_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,current_persistence,1-6h,6190,3.648142,6.990142,0.516136
1,xgboost,1-6h,6190,6.580459,9.119446,0.176454
2,hist_gradient_boosting,1-6h,6190,6.691069,9.356443,0.133093
3,previous_day_persistence,1-6h,6190,7.476559,11.597199,-0.331856
4,ridge,1-6h,6190,9.016432,11.763965,-0.370435
5,xgboost,13-24h,12163,7.150829,9.623150,0.104680
6,hist_gradient_boosting,13-24h,12163,7.160805,9.813267,0.068954
7,current_persistence,13-24h,12163,6.353498,10.294232,-0.024546
8,previous_day_persistence,13-24h,12163,7.970936,12.161715,-0.429991
9,ridge,13-24h,12163,9.461648,12.342200,-0.472750


## **3D.3** Limited XGBoost refinement

The first XGBoost model produced the strongest overall validation RMSE so far.

Instead of running a large hyperparameter search, a small number of controlled
configurations are evaluated.

The goal is to test whether modest changes to tree depth, regularization, and
learning rate improve validation performance without making the process
unnecessarily complex.

The original XGBoost model remains unchanged and is included in the comparison.

In [56]:
XGBOOST_CONFIGURATIONS = {
    "xgboost_initial": {
        "n_estimators": 1500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 10,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
    },
    "xgboost_shallower": {
        "n_estimators": 1800,
        "learning_rate": 0.04,
        "max_depth": 4,
        "min_child_weight": 15,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.2,
        "reg_lambda": 2.0,
    },
    "xgboost_regularized": {
        "n_estimators": 1800,
        "learning_rate": 0.04,
        "max_depth": 5,
        "min_child_weight": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.5,
        "reg_lambda": 3.0,
    },
}

print("Configurations to evaluate:")
for name in XGBOOST_CONFIGURATIONS:
    print("-", name)

Configurations to evaluate:
- xgboost_initial
- xgboost_shallower
- xgboost_regularized


### Train and evaluate each configuration

Every configuration uses:

- the same training split
- the same chronological validation split
- the same ordered feature columns
- the same RMSE early-stopping criterion
- the same random seed

Only a small set of model parameters changes.

In [57]:
xgboost_refinement_results = []
xgboost_refinement_models = {}
xgboost_refinement_predictions = {}

for model_name, parameters in XGBOOST_CONFIGURATIONS.items():
    print(f"\nTraining {model_name}...")

    model = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        eval_metric="rmse",
        early_stopping_rounds=75,
        random_state=42,
        n_jobs=-1,
        **parameters,
    )

    training_start = perf_counter()

    model.fit(
        X_train,
        y_train,
        eval_set=[
            (X_validation, y_validation),
        ],
        verbose=False,
    )

    training_seconds = (
        perf_counter()
        - training_start
    )

    prediction_start = perf_counter()

    predictions = model.predict(
        X_validation
    )

    prediction_seconds = (
        perf_counter()
        - prediction_start
    )

    metrics = calculate_regression_metrics(
        y_true=y_validation,
        y_pred=predictions,
    )

    negative_predictions = int(
        (predictions < 0).sum()
    )

    xgboost_refinement_results.append(
        {
            "model": model_name,
            "training_seconds": training_seconds,
            "prediction_seconds": prediction_seconds,
            "best_iteration": int(
                model.best_iteration
            ),
            "best_validation_rmse": float(
                model.best_score
            ),
            "negative_predictions": negative_predictions,
            **metrics,
        }
    )

    xgboost_refinement_models[
        model_name
    ] = model

    xgboost_refinement_predictions[
        model_name
    ] = predictions

    print(
        f"{model_name} completed. "
        f"RMSE={metrics['rmse']:.4f}, "
        f"MAE={metrics['mae']:.4f}, "
        f"R²={metrics['r2']:.4f}"
    )


Training xgboost_initial...
xgboost_initial completed. RMSE=9.6397, MAE=7.1830, R²=0.0225

Training xgboost_shallower...
xgboost_shallower completed. RMSE=9.6042, MAE=7.0802, R²=0.0297

Training xgboost_regularized...
xgboost_regularized completed. RMSE=9.7038, MAE=7.2847, R²=0.0094


In [58]:
xgboost_refinement_comparison_df = (
    pd.DataFrame(
        xgboost_refinement_results
    )
    .sort_values(
        "rmse"
    )
    .reset_index(drop=True)
)

display(
    xgboost_refinement_comparison_df
)

,model,training_seconds,prediction_seconds,best_iteration,best_validation_rmse,negative_predictions,mae,rmse,r2
0,xgboost_shallower,30.368536,0.213932,323,9.604153,0,7.080239,9.604153,0.029668
1,xgboost_initial,55.206586,0.368576,351,9.639673,0,7.182968,9.639673,0.022478
2,xgboost_regularized,25.767774,0.223983,207,9.703816,0,7.284746,9.703816,0.009425


In [59]:
best_xgboost_model_name = (
    xgboost_refinement_comparison_df
    .iloc[0]["model"]
)

best_xgboost_model = (
    xgboost_refinement_models[
        best_xgboost_model_name
    ]
)

best_xgboost_validation_predictions = (
    xgboost_refinement_predictions[
        best_xgboost_model_name
    ]
)

print(
    "Best XGBoost configuration:",
    best_xgboost_model_name,
)

print(
    "Best validation RMSE:",
    xgboost_refinement_comparison_df
    .iloc[0]["rmse"],
)

Best XGBoost configuration: xgboost_shallower
Best validation RMSE: 9.604153356925021


In [61]:
best_xgboost_validation_results_df = (
    validation_df[
        [
            "reference_time",
            "target_time",
            "forecast_horizon_hours",
            TARGET_COLUMN,
        ]
    ]
    .copy()
)

best_xgboost_validation_results_df[
    "prediction"
] = best_xgboost_validation_predictions

best_xgboost_validation_results_df[
    "horizon_group"
] = (
    best_xgboost_validation_results_df[
        "forecast_horizon_hours"
    ]
    .map(assign_horizon_group)
)

In [62]:
best_xgboost_group_records = []

for group_name in HORIZON_GROUPS:
    group_df = (
        best_xgboost_validation_results_df.loc[
            best_xgboost_validation_results_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    group_metrics = calculate_regression_metrics(
        y_true=group_df[TARGET_COLUMN],
        y_pred=group_df["prediction"],
    )

    best_xgboost_group_records.append(
        {
            "model": best_xgboost_model_name,
            "horizon_group": group_name,
            "rows": len(group_df),
            **group_metrics,
        }
    )

best_xgboost_group_metrics_df = pd.DataFrame(
    best_xgboost_group_records
)

display(
    best_xgboost_group_metrics_df
)

,model,horizon_group,rows,mae,rmse,r2
0,xgboost_shallower,1-6h,6190,6.476529,8.882554,0.218684
1,xgboost_shallower,7-12h,6129,6.746634,9.239502,0.166199
2,xgboost_shallower,13-24h,12163,7.117727,9.692964,0.091642
3,xgboost_shallower,25-48h,23709,7.168849,9.669828,-0.050194
4,xgboost_shallower,49-72h,23065,7.220054,9.769384,-0.031042


### Validate a short-horizon hybrid strategy

Validation results show that current-value persistence performs strongly for
short horizons, while XGBoost performs better for medium and long horizons.

A simple hybrid strategy is therefore evaluated:

- horizons 1–12 use current-value persistence
- horizons 13–72 use the best XGBoost prediction

This is still a deterministic forecasting rule and can be implemented easily
during inference.

In [63]:
hybrid_validation_predictions = np.where(
    validation_df[
        "forecast_horizon_hours"
    ].to_numpy() <= 12,
    validation_df[
        "pm25_current"
    ].to_numpy(),
    best_xgboost_validation_predictions,
)

In [64]:
hybrid_validation_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=hybrid_validation_predictions,
    )
)

hybrid_overall_metrics_df = pd.DataFrame(
    [
        {
            "model": "hybrid_persistence_1_12_xgboost_13_72",
            **hybrid_validation_metrics,
        }
    ]
)

display(
    hybrid_overall_metrics_df
)

,model,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_13_72,6.695642,9.419553,0.066611


In [65]:
final_validation_comparison_df = (
    pd.concat(
        [
            validation_model_comparison_df[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
            xgboost_refinement_comparison_df[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
            hybrid_overall_metrics_df[
                [
                    "model",
                    "mae",
                    "rmse",
                    "r2",
                ]
            ],
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["model"],
        keep="last",
    )
    .sort_values("rmse")
    .reset_index(drop=True)
)

display(
    final_validation_comparison_df
)

,model,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_13_72,6.695642,9.419553,0.066611
1,xgboost_shallower,7.080239,9.604153,0.029668
2,xgboost_initial,7.182968,9.639673,0.022478
3,xgboost,7.182968,9.639673,0.022478
4,xgboost_regularized,7.284746,9.703816,0.009425
5,hist_gradient_boosting,7.176154,9.827731,-0.016035
6,current_persistence,6.621121,10.506971,-0.161334
7,previous_day_persistence,7.942474,11.887120,-0.486467
8,ridge,9.779916,12.994571,-0.776340


## **3E.** Validation model selection

The validation comparison is complete.

The selected forecasting strategy is:

- current-value persistence for horizons 1 through 12
- `xgboost_shallower` for horizons 13 through 72

This strategy achieved the lowest overall validation RMSE among all evaluated
approaches.

It was selected because:

- persistence performed best for short horizons
- XGBoost performed best for medium and long horizons
- the hybrid had the strongest overall validation RMSE
- XGBoost produced no negative PM2.5 predictions
- the routing rule is simple to reproduce during inference

The untouched test dataset has not been used for this decision.

In [66]:
SELECTED_STRATEGY_NAME = (
    "hybrid_persistence_1_12_"
    "xgboost_shallower_13_72"
)

SELECTED_XGBOOST_MODEL_NAME = (
    "xgboost_shallower"
)

HYBRID_PERSISTENCE_MAX_HORIZON = 12

selected_validation_metrics = {
    "strategy": SELECTED_STRATEGY_NAME,
    "mae": float(
        hybrid_overall_metrics_df.iloc[0]["mae"]
    ),
    "rmse": float(
        hybrid_overall_metrics_df.iloc[0]["rmse"]
    ),
    "r2": float(
        hybrid_overall_metrics_df.iloc[0]["r2"]
    ),
    "persistence_horizons": "1-12",
    "xgboost_horizons": "13-72",
}

selected_validation_metrics

{'strategy': 'hybrid_persistence_1_12_xgboost_shallower_13_72',
 'mae': 6.695642304681119,
 'rmse': 9.419552655549651,
 'r2': 0.06661111029588762,
 'persistence_horizons': '1-12',
 'xgboost_horizons': '13-72'}

## **3F.** Focused validation error analysis

Before evaluating the untouched test set, a small error analysis is performed
on the selected validation strategy.

The purpose is to identify obvious weaknesses without adding unnecessary
experiments.

The analysis checks:

- residual bias
- largest prediction errors
- error by forecast horizon
- error by target hour of day

In [67]:
selected_validation_results_df = (
    validation_df[
        [
            "reference_time",
            "target_time",
            "forecast_horizon_hours",
            TARGET_COLUMN,
            "pm25_current",
        ]
    ]
    .copy()
)

selected_validation_results_df[
    "prediction"
] = hybrid_validation_predictions

selected_validation_results_df[
    "residual"
] = (
    selected_validation_results_df[TARGET_COLUMN]
    - selected_validation_results_df["prediction"]
)

selected_validation_results_df[
    "absolute_error"
] = (
    selected_validation_results_df[
        "residual"
    ].abs()
)

selected_validation_results_df[
    "target_hour"
] = (
    selected_validation_results_df[
        "target_time"
    ].dt.hour
)

In [68]:
display(
    selected_validation_results_df[
        "residual"
    ]
    .describe()
    .to_frame(name="residual")
)

print(
    "Mean residual:",
    selected_validation_results_df[
        "residual"
    ].mean(),
)

,residual
count,71256.000000
mean,-3.215828
std,8.853673
min,-71.800000
25%,-7.557520
50%,-2.906103
75%,1.631455
max,66.600000


Mean residual: -3.215828390017133


In [69]:
largest_validation_errors_df = (
    selected_validation_results_df
    .nlargest(
        20,
        "absolute_error",
    )
    [
        [
            "reference_time",
            "target_time",
            "forecast_horizon_hours",
            TARGET_COLUMN,
            "prediction",
            "residual",
            "absolute_error",
        ]
    ]
)

display(largest_validation_errors_df)

,reference_time,target_time,forecast_horizon_hours,target_pm25_ug_m3,prediction,residual,absolute_error
15162,2026-04-02 17:00:00+00:00,2026-04-02 22:00:00+00:00,5,0.2,72.000000,-71.800000,71.800000
14045,2026-04-01 06:00:00+00:00,2026-04-03 01:00:00+00:00,43,25.9,97.196342,-71.296342,71.296342
14742,2026-04-01 23:00:00+00:00,2026-04-03 01:00:00+00:00,26,25.9,97.175018,-71.275018,71.275018
12501,2026-03-31 05:00:00+00:00,2026-04-03 01:00:00+00:00,68,25.9,96.927956,-71.027956,71.027956
12572,2026-03-31 06:00:00+00:00,2026-04-03 01:00:00+00:00,67,25.9,96.927956,-71.027956,71.027956
14094,2026-04-01 07:00:00+00:00,2026-04-03 01:00:00+00:00,42,25.9,96.909241,-71.009241,71.009241
12430,2026-03-31 04:00:00+00:00,2026-04-03 01:00:00+00:00,69,25.9,96.450409,-70.550409,70.550409
12288,2026-03-31 02:00:00+00:00,2026-04-03 01:00:00+00:00,71,25.9,95.608200,-69.708200,69.708200
12217,2026-03-31 01:00:00+00:00,2026-04-03 01:00:00+00:00,72,25.9,95.227890,-69.327890,69.327890
12359,2026-03-31 03:00:00+00:00,2026-04-03 01:00:00+00:00,70,25.9,95.227890,-69.327890,69.327890


In [70]:
validation_error_by_horizon_df = (
    selected_validation_results_df
    .groupby(
        "forecast_horizon_hours"
    )
    .agg(
        rows=(
            TARGET_COLUMN,
            "size",
        ),
        mae=(
            "absolute_error",
            "mean",
        ),
        mean_residual=(
            "residual",
            "mean",
        ),
    )
    .reset_index()
)

display(validation_error_by_horizon_df)

,forecast_horizon_hours,rows,mae,mean_residual
0,1,1039,2.061886,-0.022233
1,2,1035,3.007246,-0.045700
2,3,1032,3.668605,-0.084302
3,4,1030,4.047670,-0.093301
4,5,1028,4.456031,-0.081323
...,...,...,...,...
67,68,954,7.266139,-3.678222
68,69,953,7.264290,-3.664681
69,70,952,7.260142,-3.619582
70,71,951,7.239835,-3.602853


In [71]:
validation_error_by_target_hour_df = (
    selected_validation_results_df
    .groupby("target_hour")
    .agg(
        rows=(
            TARGET_COLUMN,
            "size",
        ),
        mae=(
            "absolute_error",
            "mean",
        ),
        mean_residual=(
            "residual",
            "mean",
        ),
    )
    .reset_index()
)

display(validation_error_by_target_hour_df)

,target_hour,rows,mae,mean_residual
0,0,3045,8.224970,-4.955439
1,1,3048,7.583971,-3.486796
2,2,3051,6.123655,-1.982617
3,3,2982,5.520467,-0.417262
4,4,2985,6.653025,-0.342999
5,5,2990,6.506462,-2.190129
6,6,3019,6.627640,-2.615566
7,7,3006,6.117911,-2.679952
8,8,2877,6.620415,-3.732254
9,9,2881,6.469831,-3.577909


## **3G.** Final untouched test evaluation

The forecasting strategy was selected using validation data only:

- horizons 1–12 use current-value persistence
- horizons 13–72 use `xgboost_shallower`

The untouched test dataset is now evaluated once to estimate how this strategy
performs on the latest historical period.

No model parameters, feature definitions, or routing thresholds will be changed
after reviewing the test results.

The evaluation reports:

- overall MAE, RMSE, and R²
- persistence baseline performance
- metrics by forecast-horizon group
- metrics for each horizon from 1 through 72
- comparison between validation and test performance

In [72]:
X_test = test_df[MODEL_FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

assert X_test.columns.tolist() == MODEL_FEATURE_COLUMNS
assert X_test.isna().sum().sum() == 0
assert y_test.isna().sum() == 0
assert len(X_test) == len(y_test)

print("Test feature shape:", X_test.shape)
print("Test target shape:", y_test.shape)
print("Test model inputs validated.")

Test feature shape: (76813, 56)
Test target shape: (76813,)
Test model inputs validated.


In [73]:
best_xgboost_model_name
best_xgboost_model

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.85
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",75
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [74]:
test_prediction_start = perf_counter()

best_xgboost_test_predictions = (
    best_xgboost_model.predict(X_test)
)

test_prediction_seconds = (
    perf_counter()
    - test_prediction_start
)

print(
    "XGBoost test predictions:",
    len(best_xgboost_test_predictions),
)

print(
    f"Test prediction completed in "
    f"{test_prediction_seconds:.2f} seconds."
)

print(
    "Negative XGBoost test predictions:",
    int((best_xgboost_test_predictions < 0).sum()),
)

XGBoost test predictions: 76813
Test prediction completed in 0.43 seconds.
Negative XGBoost test predictions: 0


In [75]:
current_persistence_test_predictions = (
    test_df["pm25_current"].to_numpy()
)

previous_day_test_predictions = (
    test_df["pm25_lag_24h"].to_numpy()
)

hybrid_test_predictions = np.where(
    test_df[
        "forecast_horizon_hours"
    ].to_numpy() <= HYBRID_PERSISTENCE_MAX_HORIZON,
    current_persistence_test_predictions,
    best_xgboost_test_predictions,
)

In [76]:
short_horizon_test_mask = (
    test_df["forecast_horizon_hours"]
    <= HYBRID_PERSISTENCE_MAX_HORIZON
)

long_horizon_test_mask = (
    ~short_horizon_test_mask
)

assert np.array_equal(
    hybrid_test_predictions[
        short_horizon_test_mask.to_numpy()
    ],
    current_persistence_test_predictions[
        short_horizon_test_mask.to_numpy()
    ],
)

assert np.array_equal(
    hybrid_test_predictions[
        long_horizon_test_mask.to_numpy()
    ],
    best_xgboost_test_predictions[
        long_horizon_test_mask.to_numpy()
    ],
)

assert np.isfinite(hybrid_test_predictions).all()

print("Hybrid test prediction routing validated.")

Hybrid test prediction routing validated.


In [77]:
test_prediction_sets = {
    "current_persistence": (
        current_persistence_test_predictions
    ),
    "previous_day_persistence": (
        previous_day_test_predictions
    ),
    best_xgboost_model_name: (
        best_xgboost_test_predictions
    ),
    SELECTED_STRATEGY_NAME: (
        hybrid_test_predictions
    ),
}

test_overall_metric_records = []

for model_name, predictions in (
    test_prediction_sets.items()
):
    metrics = calculate_regression_metrics(
        y_true=y_test,
        y_pred=predictions,
    )

    test_overall_metric_records.append(
        {
            "model": model_name,
            **metrics,
        }
    )

test_overall_metrics_df = (
    pd.DataFrame(test_overall_metric_records)
    .sort_values("rmse")
    .reset_index(drop=True)
)

display(test_overall_metrics_df)

,model,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,3.715507,4.975563,-0.003986
1,xgboost_shallower,3.869165,5.108202,-0.058228
2,current_persistence,3.835369,5.283099,-0.131933
3,previous_day_persistence,4.543849,6.117524,-0.517730


In [78]:
test_predictions_df = (
    test_df[
        [
            "reference_time",
            "target_time",
            "forecast_horizon_hours",
            TARGET_COLUMN,
            "pm25_current",
            "pm25_lag_24h",
        ]
    ]
    .copy()
)

test_predictions_df[
    "xgboost_prediction"
] = best_xgboost_test_predictions

test_predictions_df[
    "prediction"
] = hybrid_test_predictions

test_predictions_df[
    "prediction_source"
] = np.where(
    test_predictions_df[
        "forecast_horizon_hours"
    ].le(HYBRID_PERSISTENCE_MAX_HORIZON),
    "current_persistence",
    best_xgboost_model_name,
)

test_predictions_df[
    "residual"
] = (
    test_predictions_df[TARGET_COLUMN]
    - test_predictions_df["prediction"]
)

test_predictions_df[
    "absolute_error"
] = (
    test_predictions_df["residual"].abs()
)

test_predictions_df[
    "horizon_group"
] = (
    test_predictions_df[
        "forecast_horizon_hours"
    ].map(assign_horizon_group)
)

display(test_predictions_df.head())

,reference_time,target_time,forecast_horizon_hours,target_pm25_ug_m3,pm25_current,pm25_lag_24h,xgboost_prediction,prediction,prediction_source,residual,absolute_error,horizon_group
0,2026-05-30 22:00:00+00:00,2026-05-30 23:00:00+00:00,1,11.2,9.5,6.2,20.552128,9.500000,current_persistence,1.700000,1.700000,1-6h
1,2026-05-30 22:00:00+00:00,2026-05-31 00:00:00+00:00,2,10.6,9.5,6.2,19.129326,9.500000,current_persistence,1.100000,1.100000,1-6h
2,2026-05-30 22:00:00+00:00,2026-05-31 01:00:00+00:00,3,11.1,9.5,6.2,20.616108,9.500000,current_persistence,1.600000,1.600000,1-6h
3,2026-05-30 22:00:00+00:00,2026-05-31 02:00:00+00:00,4,12.0,9.5,6.2,18.339737,9.500000,current_persistence,2.500000,2.500000,1-6h
4,2026-05-30 22:00:00+00:00,2026-05-31 15:00:00+00:00,17,8.0,9.5,6.2,13.475067,13.475067,xgboost_shallower,-5.475067,5.475067,13-24h


### Test performance by forecast distance

The hybrid strategy should preserve its intended behavior:

- short horizons use persistence
- medium and long horizons use XGBoost

Metrics are calculated across the same five horizon groups used during
validation.

In [79]:
test_group_metric_records = []

for model_name, predictions in (
    test_prediction_sets.items()
):
    temporary_results_df = test_df[
        [
            "forecast_horizon_hours",
            TARGET_COLUMN,
        ]
    ].copy()

    temporary_results_df[
        "prediction"
    ] = predictions

    temporary_results_df[
        "horizon_group"
    ] = temporary_results_df[
        "forecast_horizon_hours"
    ].map(assign_horizon_group)

    for group_name in HORIZON_GROUPS:
        group_df = temporary_results_df.loc[
            temporary_results_df[
                "horizon_group"
            ].eq(group_name)
        ]

        group_metrics = calculate_regression_metrics(
            y_true=group_df[TARGET_COLUMN],
            y_pred=group_df["prediction"],
        )

        test_group_metric_records.append(
            {
                "model": model_name,
                "horizon_group": group_name,
                "rows": len(group_df),
                **group_metrics,
            }
        )

test_group_metrics_df = (
    pd.DataFrame(test_group_metric_records)
    .sort_values(
        [
            "horizon_group",
            "rmse",
        ]
    )
    .reset_index(drop=True)
)

display(test_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,current_persistence,1-6h,6632,2.389882,3.368229,0.518358
1,hybrid_persistence_1_12_xgboost_shallower_13_72,1-6h,6632,2.389882,3.368229,0.518358
2,xgboost_shallower,1-6h,6632,3.619871,4.747508,0.043132
3,previous_day_persistence,1-6h,6632,3.924849,5.510887,-0.289329
4,xgboost_shallower,13-24h,12992,3.846434,5.019546,-0.015653
5,hybrid_persistence_1_12_xgboost_shallower_13_72,13-24h,12992,3.846434,5.019546,-0.015653
6,current_persistence,13-24h,12992,3.576001,5.098127,-0.047702
7,previous_day_persistence,13-24h,12992,4.187808,5.650284,-0.286936
8,xgboost_shallower,25-48h,25595,3.884136,5.143997,-0.063744
9,hybrid_persistence_1_12_xgboost_shallower_13_72,25-48h,25595,3.884136,5.143997,-0.063744


In [80]:
selected_test_group_metrics_df = (
    test_group_metrics_df.loc[
        test_group_metrics_df["model"].eq(
            SELECTED_STRATEGY_NAME
        )
    ]
    .reset_index(drop=True)
)

display(selected_test_group_metrics_df)

,model,horizon_group,rows,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,1-6h,6632,2.389882,3.368229,0.518358
1,hybrid_persistence_1_12_xgboost_shallower_13_72,13-24h,12992,3.846434,5.019546,-0.015653
2,hybrid_persistence_1_12_xgboost_shallower_13_72,25-48h,25595,3.884136,5.143997,-0.063744
3,hybrid_persistence_1_12_xgboost_shallower_13_72,49-72h,25048,3.968784,5.265653,-0.112609
4,hybrid_persistence_1_12_xgboost_shallower_13_72,7-12h,6546,3.170196,4.408933,0.176504


In [81]:
test_per_horizon_metric_records = []

for horizon in range(1, 73):
    horizon_df = test_predictions_df.loc[
        test_predictions_df[
            "forecast_horizon_hours"
        ].eq(horizon)
    ]

    horizon_metrics = calculate_regression_metrics(
        y_true=horizon_df[TARGET_COLUMN],
        y_pred=horizon_df["prediction"],
    )

    test_per_horizon_metric_records.append(
        {
            "model": SELECTED_STRATEGY_NAME,
            "forecast_horizon_hours": horizon,
            "rows": len(horizon_df),
            **horizon_metrics,
        }
    )

test_per_horizon_metrics_df = pd.DataFrame(
    test_per_horizon_metric_records
)

display(test_per_horizon_metrics_df)

,model,forecast_horizon_hours,rows,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,1,1115,1.463498,2.112300,0.811506
1,hybrid_persistence_1_12_xgboost_shallower_13_72,2,1110,1.975225,2.797085,0.668619
2,hybrid_persistence_1_12_xgboost_shallower_13_72,3,1106,2.370796,3.314821,0.533014
3,hybrid_persistence_1_12_xgboost_shallower_13_72,4,1103,2.660199,3.662579,0.429849
4,hybrid_persistence_1_12_xgboost_shallower_13_72,5,1100,2.871000,3.882386,0.358784
...,...,...,...,...,...,...
67,hybrid_persistence_1_12_xgboost_shallower_13_72,68,1036,4.039638,5.379598,-0.103069
68,hybrid_persistence_1_12_xgboost_shallower_13_72,69,1035,4.052466,5.408955,-0.095450
69,hybrid_persistence_1_12_xgboost_shallower_13_72,70,1034,4.064004,5.426136,-0.090756
70,hybrid_persistence_1_12_xgboost_shallower_13_72,71,1034,4.083856,5.449091,-0.094236


### Compare validation and test performance

Validation metrics were used to select the strategy.

Test metrics represent its final performance on unseen chronological data.

A meaningful performance decline may indicate temporal distribution shift or
some validation overfitting. A small difference is expected.

In [82]:
selected_test_metrics_row = (
    test_overall_metrics_df.loc[
        test_overall_metrics_df["model"].eq(
            SELECTED_STRATEGY_NAME
        )
    ]
    .iloc[0]
)

validation_test_comparison_df = pd.DataFrame(
    [
        {
            "dataset": "validation",
            "mae": selected_validation_metrics["mae"],
            "rmse": selected_validation_metrics["rmse"],
            "r2": selected_validation_metrics["r2"],
        },
        {
            "dataset": "test",
            "mae": float(
                selected_test_metrics_row["mae"]
            ),
            "rmse": float(
                selected_test_metrics_row["rmse"]
            ),
            "r2": float(
                selected_test_metrics_row["r2"]
            ),
        },
    ]
)

display(validation_test_comparison_df)

,dataset,mae,rmse,r2
0,validation,6.695642,9.419553,0.066611
1,test,3.715507,4.975563,-0.003986


In [83]:
test_rmse_change_percent = (
    (
        selected_test_metrics_row["rmse"]
        - selected_validation_metrics["rmse"]
    )
    / selected_validation_metrics["rmse"]
    * 100
)

print(
    "Test RMSE change relative to validation:",
    f"{test_rmse_change_percent:.2f}%",
)

Test RMSE change relative to validation: -47.18%


In [84]:
current_test_metrics_row = (
    test_overall_metrics_df.loc[
        test_overall_metrics_df["model"].eq(
            "current_persistence"
        )
    ]
    .iloc[0]
)

test_rmse_improvement_over_persistence = (
    (
        current_test_metrics_row["rmse"]
        - selected_test_metrics_row["rmse"]
    )
    / current_test_metrics_row["rmse"]
    * 100
)

print(
    "Hybrid test RMSE improvement over "
    "current persistence:",
    f"{test_rmse_improvement_over_persistence:.2f}%",
)

Hybrid test RMSE improvement over current persistence: 5.82%


## **3H.** Save model artifacts and evaluation reports

The selected forecasting strategy consists of:

- current-value persistence for horizons 1 through 12
- the trained `xgboost_shallower` model for horizons 13 through 72

The XGBoost estimator, routing configuration, ordered feature contract,
evaluation metrics, predictions, and model-selection details are now saved as
local artifacts.

These files provide the handoff between model development and the later
inference pipeline.

The artifact directory may later be uploaded to a model registry such as
Hopsworks, but model registration is not required to complete this phase.

In [85]:
from datetime import datetime, timezone
import json
import platform

import joblib
import sklearn
import xgboost

In [86]:
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


BEST_MODEL_PATH = (
    MODELS_DIR
    / "best_model.joblib"
)

MODEL_FEATURE_COLUMNS_PATH = (
    MODELS_DIR
    / "model_feature_columns.json"
)

MODEL_METADATA_PATH = (
    MODELS_DIR
    / "model_metadata.json"
)

MODEL_SELECTION_REPORT_PATH = (
    MODELS_DIR
    / "model_selection_report.json"
)


VALIDATION_COMPARISON_PATH = (
    REPORTS_DIR
    / "validation_model_comparison.csv"
)

TEST_METRICS_PATH = (
    REPORTS_DIR
    / "test_metrics.json"
)

METRICS_BY_HORIZON_PATH = (
    REPORTS_DIR
    / "metrics_by_horizon.csv"
)

METRICS_BY_HORIZON_GROUP_PATH = (
    REPORTS_DIR
    / "metrics_by_horizon_group.csv"
)

TEST_PREDICTIONS_PATH = (
    REPORTS_DIR
    / "test_predictions.parquet"
)


print("Models directory:", MODELS_DIR)
print("Reports directory:", REPORTS_DIR)

Models directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/models
Reports directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports


In [87]:
joblib.dump(
    best_xgboost_model,
    BEST_MODEL_PATH,
)

print("Saved model:", BEST_MODEL_PATH)
print("Model size:", BEST_MODEL_PATH.stat().st_size, "bytes")

Saved model: /home/riyan/Riyan/projects/pearls-aqi-predictor/models/best_model.joblib
Model size: 677236 bytes


In [88]:
model_feature_contract = {
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "feature_count": len(MODEL_FEATURE_COLUMNS),
    "target_column": TARGET_COLUMN,
    "identifier_columns": IDENTIFIER_COLUMNS,
    "forecast_horizon_feature": "forecast_horizon_hours",
    "ordered_features_required": True,
}

with open(
    MODEL_FEATURE_COLUMNS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_feature_contract,
        file,
        indent=2,
    )

print(
    "Saved model feature contract:",
    MODEL_FEATURE_COLUMNS_PATH,
)

Saved model feature contract: /home/riyan/Riyan/projects/pearls-aqi-predictor/models/model_feature_columns.json


In [89]:
selected_test_metrics = {
    "mae": float(
        selected_test_metrics_row["mae"]
    ),
    "rmse": float(
        selected_test_metrics_row["rmse"]
    ),
    "r2": float(
        selected_test_metrics_row["r2"]
    ),
}

current_persistence_test_metrics = {
    "mae": float(
        current_test_metrics_row["mae"]
    ),
    "rmse": float(
        current_test_metrics_row["rmse"]
    ),
    "r2": float(
        current_test_metrics_row["r2"]
    ),
}

previous_day_test_metrics_row = (
    test_overall_metrics_df.loc[
        test_overall_metrics_df["model"].eq(
            "previous_day_persistence"
        )
    ]
    .iloc[0]
)

previous_day_test_metrics = {
    "mae": float(
        previous_day_test_metrics_row["mae"]
    ),
    "rmse": float(
        previous_day_test_metrics_row["rmse"]
    ),
    "r2": float(
        previous_day_test_metrics_row["r2"]
    ),
}

In [91]:
validation_model_comparison_df.to_csv(
    VALIDATION_COMPARISON_PATH,
    index=False,
)

print(
    "Saved validation comparison:",
    VALIDATION_COMPARISON_PATH,
)

Saved validation comparison: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/validation_model_comparison.csv


In [92]:
test_metrics_report = {
    "selected_strategy": SELECTED_STRATEGY_NAME,
    "xgboost_model": best_xgboost_model_name,
    "persistence_horizon_max": (
        HYBRID_PERSISTENCE_MAX_HORIZON
    ),
    "selected_strategy_metrics": (
        selected_test_metrics
    ),
    "baselines": {
        "current_persistence": (
            current_persistence_test_metrics
        ),
        "previous_day_persistence": (
            previous_day_test_metrics
        ),
    },
    "rmse_improvement_over_current_persistence_percent": float(
        test_rmse_improvement_over_persistence
    ),
    "test_prediction_seconds": float(
        test_prediction_seconds
    ),
    "negative_xgboost_predictions": int(
        (
            best_xgboost_test_predictions < 0
        ).sum()
    ),
}

with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        test_metrics_report,
        file,
        indent=2,
    )

print("Saved test metrics:", TEST_METRICS_PATH)

Saved test metrics: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/test_metrics.json


In [93]:
test_per_horizon_metrics_df.to_csv(
    METRICS_BY_HORIZON_PATH,
    index=False,
)

selected_test_group_metrics_df.to_csv(
    METRICS_BY_HORIZON_GROUP_PATH,
    index=False,
)

print(
    "Saved per-horizon metrics:",
    METRICS_BY_HORIZON_PATH,
)

print(
    "Saved horizon-group metrics:",
    METRICS_BY_HORIZON_GROUP_PATH,
)

Saved per-horizon metrics: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/metrics_by_horizon.csv
Saved horizon-group metrics: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/metrics_by_horizon_group.csv


In [94]:
model_metadata = {
    "project_name": "Pearls AQI Predictor",
    "forecast_description": (
        "72-hour PM2.5-based AQI forecast for the "
        "Zafar Memon DHA reference location in Karachi."
    ),
    "model_type": "XGBRegressor",
    "model_name": best_xgboost_model_name,
    "selected_strategy": SELECTED_STRATEGY_NAME,
    "routing": {
        "horizons_1_to_12": (
            "current_pm25_persistence"
        ),
        "horizons_13_to_72": (
            best_xgboost_model_name
        ),
        "persistence_max_horizon": int(
            HYBRID_PERSISTENCE_MAX_HORIZON
        ),
    },
    "model_parameters": (
        best_xgboost_model.get_params()
    ),
    "best_iteration": int(
        best_xgboost_model.best_iteration
    ),
    "training_date_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "target_column": TARGET_COLUMN,
    "input_feature_count": len(
        MODEL_FEATURE_COLUMNS
    ),
    "ordered_feature_names": (
        MODEL_FEATURE_COLUMNS
    ),
    "data_ranges": {
        "train_reference_start": (
            train_df["reference_time"]
            .min()
            .isoformat()
        ),
        "train_reference_end": (
            train_df["reference_time"]
            .max()
            .isoformat()
        ),
        "validation_reference_start": (
            validation_df["reference_time"]
            .min()
            .isoformat()
        ),
        "validation_reference_end": (
            validation_df["reference_time"]
            .max()
            .isoformat()
        ),
        "test_reference_start": (
            test_df["reference_time"]
            .min()
            .isoformat()
        ),
        "test_reference_end": (
            test_df["reference_time"]
            .max()
            .isoformat()
        ),
    },
    "row_counts": {
        "train": len(train_df),
        "validation": len(validation_df),
        "test": len(test_df),
    },
    "validation_metrics": (
        selected_validation_metrics
    ),
    "final_test_metrics": (
        selected_test_metrics
    ),
    "test_baseline_metrics": {
        "current_persistence": (
            current_persistence_test_metrics
        ),
        "previous_day_persistence": (
            previous_day_test_metrics
        ),
    },
    "horizon_groups": HORIZON_GROUPS,
    "random_seed": 42,
    "artifact_paths": {
        "training_features": (
            str(TRAIN_DATASET_PATH)
        ),
        "validation_features": (
            str(VALIDATION_DATASET_PATH)
        ),
        "test_features": (
            str(TEST_DATASET_PATH)
        ),
        "model": str(BEST_MODEL_PATH),
    },
    "software_versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    },
    "limitations": [
        (
            "Historical observed target-hour weather "
            "was used as a proxy for forecast weather."
        ),
        (
            "Production inference will use Open-Meteo "
            "forecast weather, which introduces weather "
            "forecast uncertainty."
        ),
        (
            "The system forecasts PM2.5 at one validated "
            "reference location and is not a citywide "
            "multi-pollutant AQI model."
        ),
        (
            "Test R² was close to zero even though the "
            "selected strategy improved MAE and RMSE "
            "over persistence baselines."
        ),
    ],
}

with open(
    MODEL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_metadata,
        file,
        indent=2,
        default=str,
    )

print("Saved model metadata:", MODEL_METADATA_PATH)

Saved model metadata: /home/riyan/Riyan/projects/pearls-aqi-predictor/models/model_metadata.json


In [96]:
validation_models_evaluated = (
    validation_model_comparison_df[
        [
            "model",
            "mae",
            "rmse",
            "r2",
        ]
    ]
    .to_dict(orient="records")
)

model_selection_report = {
    "models_evaluated": (
        validation_models_evaluated
    ),
    "xgboost_configurations": (
        xgboost_refinement_comparison_df
        .to_dict(orient="records")
    ),
    "selected_strategy": (
        SELECTED_STRATEGY_NAME
    ),
    "selected_model": (
        best_xgboost_model_name
    ),
    "selection_reason": [
        (
            "The hybrid strategy achieved the lowest "
            "overall validation RMSE."
        ),
        (
            "Current-value persistence performed best "
            "for horizons 1 through 12."
        ),
        (
            "The shallower XGBoost configuration "
            "performed best for horizons 13 through 72."
        ),
        (
            "The routing rule is simple and suitable "
            "for deterministic inference."
        ),
        (
            "The selected XGBoost model produced no "
            "negative test predictions."
        ),
    ],
    "validation_metrics": (
        selected_validation_metrics
    ),
    "final_untouched_test_metrics": (
        selected_test_metrics
    ),
    "test_improvement_over_current_persistence": {
        "rmse_percent": float(
            test_rmse_improvement_over_persistence
        ),
    },
    "overfitting_assessment": (
        "No clear validation-to-test degradation was "
        "observed. Test errors were substantially lower, "
        "likely because the chronological test period "
        "contained more stable or lower-variance PM2.5 "
        "conditions."
    ),
    "major_error_patterns": [
        (
            "Forecast error increases with forecast "
            "horizon."
        ),
        (
            "Current persistence is strongest for "
            "immediate horizons."
        ),
        (
            "XGBoost provides greater value at medium "
            "and long horizons."
        ),
        (
            "The model may underrepresent extreme PM2.5 "
            "events because tree ensembles tend to "
            "regress predictions toward observed ranges."
        ),
    ],
    "limitations": (
        model_metadata["limitations"]
    ),
    "next_phase_recommendation": (
        "Build the live inference pipeline using the "
        "saved feature contract, latest PM2.5 history, "
        "Open-Meteo forecast weather, and the hybrid "
        "horizon-routing rule."
    ),
}

with open(
    MODEL_SELECTION_REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_selection_report,
        file,
        indent=2,
        default=str,
    )

print(
    "Saved model selection report:",
    MODEL_SELECTION_REPORT_PATH,
)

Saved model selection report: /home/riyan/Riyan/projects/pearls-aqi-predictor/models/model_selection_report.json


In [97]:
reloaded_model = joblib.load(
    BEST_MODEL_PATH
)

reload_sample_X = (
    X_test.iloc[:100]
)

original_sample_predictions = (
    best_xgboost_model.predict(
        reload_sample_X
    )
)

reloaded_sample_predictions = (
    reloaded_model.predict(
        reload_sample_X
    )
)

np.testing.assert_allclose(
    original_sample_predictions,
    reloaded_sample_predictions,
    rtol=1e-7,
    atol=1e-7,
)

print("Saved model reload validation passed.")

Saved model reload validation passed.


In [100]:
required_test_prediction_columns = [
    "reference_time",
    "target_time",
    "forecast_horizon_hours",
    TARGET_COLUMN,
    "prediction",
    "prediction_source",
    "residual",
    "absolute_error",
]

missing_test_prediction_columns = [
    column
    for column in required_test_prediction_columns
    if column not in test_predictions_df.columns
]

print(
    "Missing test prediction columns:",
    missing_test_prediction_columns,
)

assert not missing_test_prediction_columns

assert len(test_predictions_df) == len(test_df)

assert (
    test_predictions_df[
        required_test_prediction_columns
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

print("Test predictions DataFrame validated.")

Missing test prediction columns: []
Test predictions DataFrame validated.


In [101]:
TEST_PREDICTIONS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

test_predictions_df.to_parquet(
    TEST_PREDICTIONS_PATH,
    index=False,
)

print(
    "Saved test predictions:",
    TEST_PREDICTIONS_PATH,
)

print(
    "File exists:",
    TEST_PREDICTIONS_PATH.exists(),
)

print(
    "File size:",
    TEST_PREDICTIONS_PATH.stat().st_size,
    "bytes",
)

Saved test predictions: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/test_predictions.parquet
File exists: True
File size: 1754978 bytes


In [102]:
test_predictions_reloaded_df = pd.read_parquet(
    TEST_PREDICTIONS_PATH
)

print(
    "Reloaded test predictions shape:",
    test_predictions_reloaded_df.shape,
)

pd.testing.assert_frame_equal(
    test_predictions_df.reset_index(drop=True),
    test_predictions_reloaded_df.reset_index(drop=True),
    check_dtype=True,
)

print("Test predictions reload validation passed.")

Reloaded test predictions shape: (76813, 12)
Test predictions reload validation passed.


In [103]:
saved_artifact_paths = {
    "best_model": BEST_MODEL_PATH,
    "feature_contract": (
        MODEL_FEATURE_COLUMNS_PATH
    ),
    "model_metadata": MODEL_METADATA_PATH,
    "model_selection_report": (
        MODEL_SELECTION_REPORT_PATH
    ),
    "validation_comparison": (
        VALIDATION_COMPARISON_PATH
    ),
    "test_metrics": TEST_METRICS_PATH,
    "metrics_by_horizon": (
        METRICS_BY_HORIZON_PATH
    ),
    "metrics_by_horizon_group": (
        METRICS_BY_HORIZON_GROUP_PATH
    ),
    "test_predictions": (
        TEST_PREDICTIONS_PATH
    ),
}

artifact_summary_df = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "exists": artifact_path.exists(),
            "size_bytes": (
                artifact_path.stat().st_size
                if artifact_path.exists()
                else None
            ),
            "path": str(artifact_path),
        }
        for artifact_name, artifact_path
        in saved_artifact_paths.items()
    ]
)

display(artifact_summary_df)

assert artifact_summary_df["exists"].all()

print("All Phase 3 artifacts saved successfully.")

,artifact,exists,size_bytes,path
0,best_model,True,677236,/home/riyan/Riyan/projects/pearls-aqi-predicto...
1,feature_contract,True,1653,/home/riyan/Riyan/projects/pearls-aqi-predicto...
2,model_metadata,True,5609,/home/riyan/Riyan/projects/pearls-aqi-predicto...
3,model_selection_report,True,4088,/home/riyan/Riyan/projects/pearls-aqi-predicto...
4,validation_comparison,True,386,/home/riyan/Riyan/projects/pearls-aqi-predicto...
5,test_metrics,True,732,/home/riyan/Riyan/projects/pearls-aqi-predicto...
6,metrics_by_horizon,True,8181,/home/riyan/Riyan/projects/pearls-aqi-predicto...
7,metrics_by_horizon_group,True,622,/home/riyan/Riyan/projects/pearls-aqi-predicto...
8,test_predictions,True,1754978,/home/riyan/Riyan/projects/pearls-aqi-predicto...


All Phase 3 artifacts saved successfully.
